# Demo 01 — The Problem: Why Sentence-Level Citations?

*For tax auditors, compliance officers, and PMs. No code knowledge required.*

---

## The setup

Auditors and compliance reviewers are being asked to sign off on AI-generated
summaries of tax guidance. That's a high-stakes ask:

- A **document-level citation** ("this answer is based on IRS Publication 583")
  tells the reviewer *which* 30-page PDF to read. It doesn't tell them *where*.
- A **paragraph-level citation** narrows it to a section, but the reviewer
  still has to re-read the paragraph and mentally diff it against the answer.
- Neither lets the reviewer verify the answer **word-by-word**, which is what
  audit sign-off actually requires.

The cost of a wrong citation in an audit context is high: a bad sign-off can
propagate into guidance, filings, and regulatory exposure.

## The promise of this notebook

We'll take **one real question** from our cached evaluation run and answer it
two ways:

1. **Without sentence-level citations** — the way most RAG demos look today.
2. **With sentence-level citations** — the way this prototype does it.

You'll see the difference from a reviewer's seat — no code required.

> 🔌 **Zero live calls.** Everything in this notebook reads from
> `data/notebook_cache/`. No Azure, no network, no keys.

## The question we'll use

Pulled from our cached eval run (`data/notebook_cache/eval/items.jsonl`), matched
against the cached retrieval snapshot (`data/notebook_cache/retrieval/snapshot.json`).

We picked a question with a **multi-sentence answer** that draws on **two different
IRS publications** — exactly the kind of answer where sentence-level citation
earns its keep.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Markdown, display

CACHE = Path("../data/notebook_cache")

eval_items = [json.loads(l) for l in (CACHE / "eval" / "items.jsonl").open()]
snapshot = json.loads((CACHE / "retrieval" / "snapshot.json").read_text())

# Pick the item whose question matches the cached retrieval snapshot.
item = next(it for it in eval_items if it["question"] == snapshot["query"])

# Build a sentence-id -> (text, document, page) lookup from the snapshot.
sid_lookup = {}
for ch in snapshot["chunks"]:
    for s in ch.get("sentences") or []:
        sid_lookup[s["sentence_id"]] = {
            "text": s["text"],
            "document_id": ch["document_id"],
            "page": ch.get("page"),
        }
for sc in snapshot["sentence_candidates"]:
    sid_lookup.setdefault(sc["sentence_id"], {
        "text": sc["text"],
        "document_id": sc["document_id"],
        "page": sc.get("page"),
    })

display(Markdown(f"**Question:** {item['question']}"))

**Question:** What type of payment must be reported as income if received by a dependent care provider?

---

## Version A — Without sentence-level citations

This is what a traditional RAG answer looks like: a fluent paragraph, plus a
list of source documents. Read it like an auditor would.

In [2]:
cited_docs = sorted({sid.split("-s")[0] for sid in item["pred_citation_ids"]})
doc_labels = {
    "irs_pub_17":  "IRS Publication 17 — Your Federal Income Tax for Individuals",
    "irs_pub_583": "IRS Publication 583 — Starting a Business and Keeping Records",
    "irs_pub_587": "IRS Publication 587 — Business Use of Your Home",
    "irs_pub_463": "IRS Publication 463 — Travel, Gift, and Car Expenses",
    "irs_pub_544": "IRS Publication 544 — Sales and Other Dispositions of Assets",
}
cited_doc_list = "\n".join(f"- {doc_labels.get(d, d)}" for d in cited_docs)

display(Markdown(
    f"### Answer\n\n"
    f"> {item['answer_text']}\n\n"
    f"**Cited documents:**\n\n{cited_doc_list}"
))

### Answer

> Any amount paid to you as a dependent care provider must be reported as income . If you provide childcare, either in the child's home or in your home or other place of business, the pay you receive must be included in your income . If you aren't an employee, you're probably self-employed and must include payments for your services on Schedule C (Form 1040), Profit or Loss From Business . If you're paid to babysit, even for relatives or neighborhood children, whether on a regular basis or only periodically, the rules for childcare providers apply to you . Whether you're an employee or self-employed person, your income could be subject to self-employment tax .

**Cited documents:**

- IRS Publication 17 — Your Federal Income Tax for Individuals
- IRS Publication 583 — Starting a Business and Keeping Records

### What the auditor sees

A fluent answer and a list of source documents. That's it.

To verify each claim, the reviewer has to open each publication and read it
end-to-end, keeping the answer in their head and mentally matching sentence by
sentence. For a 30-page pub, that's realistically **20+ minutes per question**.
Across a 500-question audit sample, that's on the order of **~160 hours** of
grinding verification — and the reviewer's confidence decays with every page.

And worse: if *one* sentence in the answer is subtly wrong — a paraphrase that
drifted, a figure transposed — there's no structural signal. The reviewer has
to catch it unaided.

---

## Version B — With sentence-level citations

Same question. Same underlying sources. But now every sentence in the answer
carries an inline citation marker like `[s1]`, `[s2]`, … and each marker maps
to an exact source sentence in an exact publication on an exact page.

In [3]:
# Split answer_text into sentences and attach [sN] markers.
# The eval cache stores pred_citation_ids in answer-sentence order.
raw = item["answer_text"].strip()
# Sentences in the cache are separated by " . " (space-dot-space).
parts = [p.strip() for p in raw.split(" . ") if p.strip()]
answer_sentences = [p if p.endswith(".") else p + "." for p in parts]

pred_ids = item["pred_citation_ids"]
n = min(len(answer_sentences), len(pred_ids))

marked = [f"{answer_sentences[i]} **[s{i+1}]**" for i in range(n)]
for extra in answer_sentences[n:]:
    marked.append(extra)
answer_with_citations = " ".join(marked)

display(Markdown(f"### Answer (with inline citations)\n\n> {answer_with_citations}"))

### Answer (with inline citations)

> Any amount paid to you as a dependent care provider must be reported as income. **[s1]** If you provide childcare, either in the child's home or in your home or other place of business, the pay you receive must be included in your income. **[s2]** If you aren't an employee, you're probably self-employed and must include payments for your services on Schedule C (Form 1040), Profit or Loss From Business. **[s3]** If you're paid to babysit, even for relatives or neighborhood children, whether on a regular basis or only periodically, the rules for childcare providers apply to you. **[s4]** Whether you're an employee or self-employed person, your income could be subject to self-employment tax . **[s5]**

In [4]:
rows = []
for i in range(n):
    sid = pred_ids[i]
    src = sid_lookup.get(sid, {})
    rows.append({
        "marker":    f"[s{i+1}]",
        "source_id": sid,
        "document":  doc_labels.get(src.get("document_id", ""), src.get("document_id", "?")),
        "page":      src.get("page", "?"),
        "source_sentence": src.get("text", "(source sentence not in cache snapshot)"),
    })

citation_df = pd.DataFrame(rows)
display(Markdown("### Citation map — each marker points to an exact sentence"))
display(citation_df)

### Citation map — each marker points to an exact sentence

,marker,source_id,document,page,source_sentence
0,[s1],irs_pub_17-s04328,IRS Publication 17 — Your Federal Income Tax f...,49,"If you provide childcare, either in the child'..."
1,[s2],irs_pub_17-s04329,IRS Publication 17 — Your Federal Income Tax f...,49,"If you aren't an employee, you're probably sel..."
2,[s3],irs_pub_17-s04332,IRS Publication 17 — Your Federal Income Tax f...,49,"If you're paid to babysit, even for relatives ..."
3,[s4],irs_pub_17-s04334,IRS Publication 17 — Your Federal Income Tax f...,49,Whether you're an em- ployee or self-employed ...
4,[s5],irs_pub_583-s00266,IRS Publication 583 — Starting a Business and ...,4,Any amount paid to you as a dependent care pro...


### What the auditor sees now

Every claim carries a numbered citation. The reviewer's workflow shifts from
*"read the whole publication"* to *"click `[s3]`, read one sentence, check the
box."*

- **Wrong citation?** The sentence won't match the claim — it's obvious.
- **Missing citation?** A claim with no `[sN]` marker is flagged before the
  reviewer ever opens the source.
- **Cross-document answer?** The markers make the hand-off between publications
  explicit (note how this answer draws on both Pub 17 and Pub 583).

---

## What this capability unlocks

- **Time per question**: sentence-by-sentence verification in seconds, not
  minutes. The reviewer jumps straight to the cited sentence.
- **Confidence**: structural markers (missing citation, mismatched citation)
  catch classes of error the reviewer would otherwise have to spot unaided.
- **Downstream trust**: once sentence-level grounding is reliable, it becomes
  the foundation for more agentic workflows — summarization, cross-referencing,
  draft guidance — because every claim is traceable back to source.

> ⚠️ **Caveat.** This notebook is demonstrating the *capability*, not a
> validated productivity study. The "20+ minutes" and "~160 hours" figures
> above are illustrative of the shape of the problem, not measured results.
> Customer validation of the time/confidence delta is **Phase 1b** work.

---

## Where to go next

- 👉 **`demo_02_pipeline_tour.ipynb`** — a tour of how the pipeline actually
  produces sentence-level citations end-to-end. No Azure calls required; it
  replays from the same cache you just saw.
- 📊 **`demo_03_metrics_that_matter.ipynb`** — what we measure (precision,
  recall, faithfulness, stability) and why those metrics are the right ones
  for audit sign-off.